In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.simulation.order_stream import OrderStream
from src.simulation.simulator import Simulator
from src.dynamicProgramming.value_function import ValueFunction
from src.dynamicProgramming.dp_scheduler import DPScheduler
from src.dynamicProgramming.insertion_policy import GreedyInsertionPolicy

from geopy.distance import geodesic

data_path = Path("../data/features/olist/delivery_jobs_dataset.csv")

In [13]:
# loading a sample dataset
jobs = pd.read_csv(
    data_path,
    parse_dates = [
        'ready_time',
        'due_date'
    ]
)

# dropping rows without lat & lng coordinates
jobs = jobs.dropna(
    subset = [
        'pickup_lat',
        'pickup_lng',
        'delivery_lat',
        'delivery_lng'
    ]
)

# filtering based on distance
def distance(row):
    dist = geodesic(
        (row.pickup_lat, row.pickup_lng),
        (row.delivery_lat, row.delivery_lng)
    ).km
    return dist

jobs['distance_km'] = jobs.apply(distance, axis = 1)
jobs = jobs.loc[jobs['distance_km'] <= 15]

# generating a scenario from samples
jobs = jobs.sort_values('ready_time').head(15).copy()
start = pd.Timestamp("2026-01-01 08:00:00")

jobs['ready_time'] = start + pd.to_timedelta(
    np.arange(len(jobs)) * 5,
    unit = 'm'
)

jobs['due_date'] = jobs['ready_time'] + pd.Timedelta(hours = 4)
jobs['service_time_min'] = 5
display(jobs.head())

,job_id,order_id,seller_id,customer_id,pickup_lat,pickup_lng,delivery_lat,delivery_lng,ready_time,due_date,service_time_min,demand,distance_km
66586,ed8c7b1b3eb256c70ce0c74231e1da88,ed8c7b1b3eb256c70ce0c74231e1da88,5b179e9e8cc7ab6fd113a46ca584da81,da0ba2a9935bca5b4610b0e3bca9d3b4,-23.568771,-46.698110,-23.453962,-46.731884,2026-01-01 08:00:00,2026-01-01 12:00:00,5,1.0,13.174828
31857,79ffdd52a918bbe867895a4b183d6457,79ffdd52a918bbe867895a4b183d6457,c7dcd301ecfe5ab7f778ac172cf74be7,f9808148a262b51d20e2d777eee6676c,-19.918527,-43.939890,-19.948076,-43.947618,2026-01-01 08:05:00,2026-01-01 12:05:00,5,1.0,3.369796
16233,c4b41c36dd589e901f6879f25a74ec1d,c4b41c36dd589e901f6879f25a74ec1d,ce27a3cc3c8cc1ea79d11e561e9bebb6,4bb880cac21c7a9e1371ab1ebd601706,-23.541812,-46.624550,-23.534776,-46.671765,2026-01-01 08:10:00,2026-01-01 12:10:00,5,1.0,4.883740
93439,6b3ee7697a02619a0ace2b3f0aa46bde,6b3ee7697a02619a0ace2b3f0aa46bde,ce27a3cc3c8cc1ea79d11e561e9bebb6,21a6abdf0197fbe57451bd0a1d3c59a2,-23.541812,-46.624550,-23.521920,-46.482302,2026-01-01 08:15:00,2026-01-01 12:15:00,5,1.0,14.692025
3781,3b2ca3293a7ce539ea2379d704fa37ce,3b2ca3293a7ce539ea2379d704fa37ce,dd2bdf855a9172734fbc3744021ae9b9,06a70917afd2dcd59396e1eac836c646,-19.869495,-43.950944,-19.843643,-43.904064,2026-01-01 08:20:00,2026-01-01 12:20:00,5,1.0,5.683464


In [ ]:
# initializing simulation components
stream = OrderStream(jobs)
policy = GreedyInsertionPolicy()
value_function = ValueFunction(num_features = 5)

scheduler = DPScheduler(
    insertion_policy = policy,
    value_function = value_function,
    gamma = 0.95,
    n_couriers = 5
)

simulator = Simulator(stream, scheduler)

# running simulation
state = simulator.run(
    start_time = jobs['ready_time'].min(),
    end_time = jobs['ready_time'].min() + pd.Timedelta(hours = 6),
    step_minutes = 5
)

for courier in state.couriers:
    print(f"Courier {courier.courier_id}")
    print(f"Completed jobs: {len(courier.completed_jobs)}")
    print(f"Completed job IDs: {[job['job_id'] for job in courier.completed_jobs]}")
    print("---")

Courier 0
Completed jobs: 15
Completed job IDs: ['ed8c7b1b3eb256c70ce0c74231e1da88', '79ffdd52a918bbe867895a4b183d6457', 'c4b41c36dd589e901f6879f25a74ec1d', '6b3ee7697a02619a0ace2b3f0aa46bde', '3b2ca3293a7ce539ea2379d704fa37ce', 'b2f92b2f7047cd8b35580d629d7b3bfb', '98974b076b01553d49ee6467905675a7', '59e7ca596664da0a459a145eb95cc9c5', '323dad8c483c7f8b818a825d257f4aa0', 'f7310018040436b01ab03f81b301b5de', '38c5c2886f2aab75bb1ae3f79a3f300d', 'd3a84031db6c5de813d5a3a3489712ca', '8fe36dfd03ccfe3ed3d492c146705b9d', 'deb08269080b64b8e69bcd06b808889a', 'f9c44da06151c190a9a6c9c712873d10']
---
Courier 1
Completed jobs: 0
Completed job IDs: []
---
Courier 2
Completed jobs: 0
Completed job IDs: []
---
Courier 3
Completed jobs: 0
Completed job IDs: []
---
Courier 4
Completed jobs: 0
Completed job IDs: []
---
